# 00_apply_security_rules

Aplica **column masking** y **row-level security** sobre `fintech_finpay_dev.silver.users`.

Regla del proyecto:
- Campos PII: `full_name`, `document_id`, `email`, `phone`.
- Solo el grupo/rol `ingenieria` puede ver valores reales.
- El resto de usuarios verá los campos PII enmascarados y no podrá ver filas si se aplica el row filter.

Ejecutar este notebook **después** de que el pipeline haya creado `fintech_finpay_dev.silver.users`.


In [ ]:
USE CATALOG fintech_finpay_dev;
USE SCHEMA silver;


## 1. Funciones de column masking


In [ ]:
CREATE OR REPLACE FUNCTION fintech_finpay_dev.silver.mask_pii_string(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN value
    WHEN value IS NULL THEN NULL
    ELSE '***MASKED***'
  END;


In [ ]:
CREATE OR REPLACE FUNCTION fintech_finpay_dev.silver.mask_email(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN value
    WHEN value IS NULL THEN NULL
    WHEN instr(value, '@') > 0 THEN concat('***@', split(value, '@')[1])
    ELSE '***MASKED_EMAIL***'
  END;


In [ ]:
CREATE OR REPLACE FUNCTION fintech_finpay_dev.silver.mask_phone(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN value
    WHEN value IS NULL THEN NULL
    WHEN length(value) >= 4 THEN concat('******', right(value, 4))
    ELSE '******'
  END;


## 2. Función de row-level security


In [ ]:
CREATE OR REPLACE FUNCTION fintech_finpay_dev.silver.users_row_filter(country STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('ingenieria');


## 3. Aplicar column masking sobre campos PII


In [ ]:
ALTER TABLE fintech_finpay_dev.silver.users
ALTER COLUMN full_name
SET MASK fintech_finpay_dev.silver.mask_pii_string;

ALTER TABLE fintech_finpay_dev.silver.users
ALTER COLUMN document_id
SET MASK fintech_finpay_dev.silver.mask_pii_string;

ALTER TABLE fintech_finpay_dev.silver.users
ALTER COLUMN email
SET MASK fintech_finpay_dev.silver.mask_email;

ALTER TABLE fintech_finpay_dev.silver.users
ALTER COLUMN phone
SET MASK fintech_finpay_dev.silver.mask_phone;


## 4. Aplicar row filter sobre `silver.users`


In [ ]:
ALTER TABLE fintech_finpay_dev.silver.users
SET ROW FILTER fintech_finpay_dev.silver.users_row_filter
ON (country);


## 5. Permisos mínimos para el grupo `ingenieria`

Ejecuta esta celda solo si el grupo `ingenieria` existe en el workspace/account.


In [ ]:
GRANT USE CATALOG ON CATALOG fintech_finpay_dev TO `ingenieria`;

GRANT USE SCHEMA ON SCHEMA fintech_finpay_dev.silver TO `ingenieria`;

GRANT SELECT ON TABLE fintech_finpay_dev.silver.users TO `ingenieria`;


## 6. Validación técnica


In [ ]:
DESCRIBE EXTENDED fintech_finpay_dev.silver.users;


In [ ]:
SELECT
  user_id,
  full_name,
  document_id,
  email,
  phone,
  country,
  segment,
  registration_date
FROM fintech_finpay_dev.silver.users
LIMIT 20;
